In [ ]:
import pandas as pd
import numpy as np

column_names = [
    "Sex", "Length", "Diameter", "Height",
    "WholeWeight", "ShuckedWeight", "VisceraWeight",
    "ShellWeight", "Rings"
]

data = pd.read_csv("abalone.data", header=None, names=column_names)

# Print number of rows
print("Number of rows:", len(data))

# Print column names
print("Column names:", data.columns)

# Print first 5 rows
print(data.head())

# Checkpoint:
# what is input: physical measurements of abalone
# what is output: number of rings (used to estimate age)
# why output is numeric: rings represent count → numeric regression target


Number of rows: 4177
Column names: Index(['Sex', 'Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight',
       'VisceraWeight', 'ShellWeight', 'Rings'],
      dtype='object')
  Sex  Length  Diameter  Height  WholeWeight  ShuckedWeight  VisceraWeight  \
0   M   0.455     0.365   0.095       0.5140         0.2245         0.1010   
1   M   0.350     0.265   0.090       0.2255         0.0995         0.0485   
2   F   0.530     0.420   0.135       0.6770         0.2565         0.1415   
3   M   0.440     0.365   0.125       0.5160         0.2155         0.1140   
4   I   0.330     0.255   0.080       0.2050         0.0895         0.0395   

   ShellWeight  Rings  
0        0.150     15  
1        0.070      7  
2        0.210      9  
3        0.155     10  
4        0.055      7  


In [ ]:
# Create target y = Rings + 1.5
y = data["Rings"] + 1.5


In [ ]:
# Select exactly 3 numeric features
X = data[["Length", "Diameter", "ShellWeight"]]

# Justification:
# Feature 1: Length – overall size strongly correlates with age
# Feature 2: Diameter – complements length for body volume
# Feature 3: ShellWeight – shell grows thicker/heavier with age


In [ ]:
X = X.values
y = y.values.reshape(-1, 1)

split_idx = int(0.8 * len(X))

X_train = X[:split_idx]
y_train = y[:split_idx]

X_test = X[split_idx:]
y_test = y[split_idx:]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (3341, 3)
y_train: (3341, 1)
X_test: (836, 3)
y_test: (836, 1)


In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Checkpoint:
# why normalization is needed for learning:
# Different feature scales cause unstable gradients and slow convergence.
# Normalization ensures balanced updates.


In [ ]:
def forward(X, w, b):
    """
    computes y_hat = Xw + b
    """
    y_hat = X.dot(w) + b

    print("X shape:", X.shape)
    print("w shape:", w.shape)
    print("b shape:", b)
    print("y_hat shape:", y_hat.shape)

    # Checkpoint:
    # parameters are: weights (w) and bias (b)
    # number of parameters: 3 weights + 1 bias = 4

    return y_hat


In [ ]:
def mse(y, y_hat):
    loss = np.mean((y - y_hat) ** 2)
    return loss

# Checkpoint:
# why square: penalizes larger errors more strongly
# what mistakes are expensive: large prediction deviations


Checkpoint:

what gradient means in words:
Gradient tells the direction and rate of increase of loss.

why subtracting gradient reduces loss:
Moving opposite to gradient moves toward lower loss.


In [ ]:
def grad_w(X, y, y_hat):
    N = len(y)
    dW = (2 / N) * X.T.dot(y_hat - y)
    return dW

def grad_b(y, y_hat):
    N = len(y)
    db = (2 / N) * np.sum(y_hat - y)
    return db

# Checkpoint:
# meaning of large gradient: loss changes rapidly → parameters far from optimal
# effect of too-large learning rate: divergence and exploding loss


In [ ]:
# Initialize parameters
np.random.seed(0)
w = np.random.randn(3, 1) * 0.01
b = 0.0

# Hyperparameters
lr = 0.01
epochs = 1000

# Initial expectation:
# Loss should decrease gradually, not instantly

for epoch in range(epochs):
    # 1) forward pass
    y_hat = X_train.dot(w) + b

    # 2) compute loss
    loss = mse(y_train, y_hat)

    # 3) compute gradients
    dW = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    # 4) update parameters
    w -= lr * dW
    b -= lr * db

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss}")

# Revised expectation after training:
# Loss decreases smoothly and stabilizes


Epoch 0, Loss: 144.17831736922406
Epoch 100, Loss: 9.201549523151352
Epoch 200, Loss: 6.791603706080492
Epoch 300, Loss: 6.6851394315379595
Epoch 400, Loss: 6.643926006132289
Epoch 500, Loss: 6.618599647121504
Epoch 600, Loss: 6.602358692230308
Epoch 700, Loss: 6.591561163764089
Epoch 800, Loss: 6.584050838785579
Epoch 900, Loss: 6.578546017935997


In [ ]:
# Predictions on test set
y_test_hat = X_test.dot(w) + b

# Test MSE
test_mse = mse(y_test, y_test_hat)

# Test MAE
test_mae = np.mean(np.abs(y_test - y_test_hat))

print("Test MSE:", test_mse)
print("Test MAE:", test_mae)

# Print 5 examples
print("\nSample predictions:")
for i in range(5):
    print(
        "True age:", y_test[i][0],
        "Predicted age:", y_test_hat[i][0],
        "Absolute error:", abs(y_test[i][0] - y_test_hat[i][0])
    )

# Checkpoint:
# systematic errors: higher error for very old abalones
# observed bias: model underestimates extreme ages


Test MSE: 5.117610559034677
Test MAE: 1.7243772416299994

Sample predictions:
True age: 13.5 Predicted age: 10.786728908057285 Absolute error: 2.7132710919427154
True age: 15.5 Predicted age: 9.651008710011213 Absolute error: 5.848991289988787
True age: 14.5 Predicted age: 10.061079010642512 Absolute error: 4.438920989357488
True age: 14.5 Predicted age: 11.119772007983862 Absolute error: 3.380227992016138
True age: 13.5 Predicted age: 11.43350901583535 Absolute error: 2.06649098416465
